In [2]:
import pandas as pd
from utils import *
import matplotlib.pyplot as plt
import os

### Pluviómetro

Cargamos los datos de todos los pluviómetros disponibles. Igual que en $\textit{Relación\_Pluvio-Boyas.ipynb}$

In [3]:

path = "Ramblas/"
# Diccionario para almacenar los dataframes
dataframes = {}

for folder in os.listdir(path):
    ruta_carpeta = os.path.join(path, folder)
    # Verificar que sea una carpeta
    if os.path.isdir(ruta_carpeta):
        # Buscar archivos que terminen en "-Pluviometro.parquet" en la carpeta
        for archivo in os.listdir(ruta_carpeta):
            if archivo.endswith('-Pluviometro.parquet'):
                ruta_archivo = os.path.join(ruta_carpeta, archivo)
                df = pd.read_parquet(ruta_archivo)
                # Crear el nombre del dataframe usando el nombre de la carpeta
                nombre_df = f"{folder[6:]}_pluviometro"
                # Almacenar el dataframe en el diccionario
                dataframes[nombre_df] = df

pluvios_to_keep = ["Desemboc Rbla Albujon_pluviometro", "La Puebla_pluviometro", "Pozo Estrecho_pluviometro", "Rbla Albujon_pluviometro", "El Estrecho_pluviometro"]
dataframes = {key: df for key, df in dataframes.items() if key in pluvios_to_keep}

# Dataframe vacío
combined_df = pd.DataFrame()

# Iterar sobre cada dataframe filtrado
for location, df in dataframes.items():
    df = df.set_index("Date")
    df = df.rename(columns={"Pluviometro": location})

    if combined_df.empty:
        combined_df = df
    else:
        combined_df = combined_df.join(df, how="outer")

combined_df = combined_df.reset_index()
# Convertir la columna 'Date' al formato datetime
combined_df['Date'] = pd.to_datetime(combined_df['Date'])
# Ponemos la fecha como índice
combined_df = combined_df.set_index('Date')
# Reemplazamos las '-' con NaN
combined_df.replace('-', np.nan, inplace=True)
# Reemplazamos NaN por 0's
combined_df.fillna(0, inplace=True)
# Ponemos como float los valores de los pluvios
combined_df = combined_df.astype({col: 'float' for col in combined_df.columns})
# Agrupamos los datos por día, sumando la precipitación acumulada
df_pluvio = combined_df.resample('D').sum()
# Definimos una columna que sea la suma de todos los pluvios
df_pluvio['Total'] = df_pluvio.loc[:, df_pluvio.columns].sum(axis=1)
df_pluvio = df_pluvio.reset_index()

### Nitratos/Fosfatos

In [ ]:
# Definir el nombre del archivo y las hojas de interés con sus respectivos rangos de columnas
filename = "Registro_Ramblas_MARMENOR.xlsx"
sheet_info = {
    "Caudal": range(1, 26),
    "Nitratos": range(1, 25),
    "Fosfatos": range(1, 25),
    # "Conductividad": range(1, 25),
    # "NitratosDiario": range(1, 14),
    # "FosfatosDiario": range(1, 14)
}

# Función para cargar y procesar cada hoja
def load_and_process_sheet(sheet_name, usecols):
    # Cargar el dataframe de la hoja con las columnas especificadas
    df = pd.read_excel(filename, sheet_name=sheet_name, skiprows=6, usecols=usecols).dropna(how="all") 
    # Procesar el dataframe: extraer fechas, eliminar NaNs, reemplazar valores
    df['Fecha'] = df['Fecha'].apply(extract_recent_date)
    df = df[~df['Fecha'].isna()]
    df = df.fillna(0)
    df = df.applymap(replace_strings)
    return df

# Cargar y procesar todos los dataframes en un diccionario con nombres como claves
dataframes = {name: load_and_process_sheet(name, cols) for name, cols in sheet_info.items()}

# Filtrar las columnas en el diccionario de dataframes
cols_to_keep = ["Fecha", "Desembocadura rambla Albujón", "Obra de paso bajo carretera Los Urrutia", "Rambla de las Matildes - corriente sur", "Lo Poyo"]

for name, df in dataframes.items():
    # Mantener solo las columnas especificadas que existen en cada dataframe
    existing_cols = df.columns.intersection(cols_to_keep)
    dataframes[name] = df.loc[:, existing_cols]


df1 = dataframes['Caudal'].set_index('Fecha')
df2 = dataframes['Nitratos'].set_index('Fecha')
aligned_df1, aligned_df2 = df1.align(df2, join='inner')
df_nitratos_diario = pd.DataFrame(index=aligned_df1.index)

for col in aligned_df1.columns:
    df_nitratos_diario[col] = aligned_df1[col] * aligned_df2[col] * 86400/1e6

dataframes['Nitratos Diario'] = df_nitratos_diario.reset_index()


df1 = dataframes['Caudal'].set_index('Fecha')
df2 = dataframes['Fosfatos'].set_index('Fecha')
aligned_df1, aligned_df2 = df1.align(df2, join='inner')
df_fosfatos_diario = pd.DataFrame(index=aligned_df1.index)

for col in aligned_df1.columns:
    df_fosfatos_diario[col] = aligned_df1[col] * aligned_df2[col] * 86400/1e6

dataframes['Fosfatos Diario'] = df_fosfatos_diario.reset_index()

In [ ]:
dataframes["Nitratos Diario"]['Nitratos CTD7'] = dataframes["Nitratos Diario"].loc[:, ["Desembocadura rambla Albujón", "Obra de paso bajo carretera Los Urrutia"]].sum(axis=1)
dataframes["Nitratos Diario"]['Nitratos CTD9'] = dataframes["Nitratos Diario"].loc[:, ["Rambla de las Matildes - corriente sur", "Lo Poyo"]].sum(axis=1)
dataframes["Fosfatos Diario"]['Fosfatos CTD7'] = dataframes["Fosfatos Diario"].loc[:, ["Desembocadura rambla Albujón", "Obra de paso bajo carretera Los Urrutia"]].sum(axis=1)
dataframes["Fosfatos Diario"]['Fosfatos CTD9'] = dataframes["Fosfatos Diario"].loc[:, ["Rambla de las Matildes - corriente sur", "Lo Poyo"]].sum(axis=1)
dataframes["Nitratos"]['Nitratos CTD7'] = dataframes["Nitratos"].loc[:, ["Desembocadura rambla Albujón", "Obra de paso bajo carretera Los Urrutia"]].sum(axis=1)
dataframes["Nitratos"]['Nitratos CTD9'] = dataframes["Nitratos"].loc[:, ["Rambla de las Matildes - corriente sur", "Lo Poyo"]].sum(axis=1)
dataframes["Fosfatos"]['Fosfatos CTD7'] = dataframes["Fosfatos"].loc[:, ["Desembocadura rambla Albujón", "Obra de paso bajo carretera Los Urrutia"]].sum(axis=1)
dataframes["Fosfatos"]['Fosfatos CTD9'] = dataframes["Fosfatos"].loc[:, ["Rambla de las Matildes - corriente sur", "Lo Poyo"]].sum(axis=1)

In [ ]:
### MIRAR SOLO CTD7 (los puntos que van a esa boya)